# 01 - Preprocess (All Datasets)

Bu notebook **tek dosyada** 3 farklı dataset için preprocess yapar:
- PlantVillage
- Plant Disease Detection (Kaggle: karagwaanntreasure/plant-disease-detection)
- PlantDoc (Converted)

Çıktı olarak her dataset için **tek bir `.npz` dosyası** üretir (train/val/test + class_names + normalization istatistikleri).

In [ ]:

# ---- Imports (keep it simple) ----
import os
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 1) Dataset yolları

`DATA_ROOT` altına dataset klasörlerini koy.

Örnek yapı (öneri):
```
Dataset/
  plantvillage/
    PlantVillage/ or color/ ... (sende nasıl çıktıysa)
  plant_disease_detection/
    Train/
    Validation/
    Test/
  plantdoc_converted/
    images/
      classA/
      classB/
```

Aşağıdaki path'leri kendi bilgisayarındaki klasörlere göre düzenle.

In [ ]:

# ---- Paths ----
DATA_ROOT = Path("../Dataset")   # change if needed

# 1) PlantVillage: a folder that contains class subfolders (images inside)
PLANTVILLAGE_DIR = DATA_ROOT / "plantvillage"   # change to your extracted folder
# Example alternatives:
# PLANTVILLAGE_DIR = DATA_ROOT / "plantvillage" / "color"
# PLANTVILLAGE_DIR = DATA_ROOT / "plantvillage" / "PlantVillage" / "color"

# 2) Plant Disease Detection: usually Train/Validation/Test each with class subfolders
PDD_DIR = DATA_ROOT / "plant_disease_detection"  # change
PDD_TRAIN_DIR = PDD_DIR / "Train"
PDD_VAL_DIR   = PDD_DIR / "Validation"
PDD_TEST_DIR  = PDD_DIR / "Test"

# 3) PlantDoc Converted: a folder that contains class subfolders (images inside)
PLANTDOC_CONVERTED_DIR = DATA_ROOT / "plantdoc_converted" / "images"  # change

# Output
OUT_DIR = Path("./preprocessed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR.resolve())


## 2) Genel ayarlar (hız için)

- `IMG_SIZE`: küçülttükçe eğitim hızlanır.
- `GRAYSCALE=True`: feature sayısını azaltır (hız +). 
- `MAX_PER_CLASS`: çok büyük datasetlerde eğitim çok uzamasın diye sınıf başına sınır koyar.

In [ ]:

# ---- Speed / size settings ----
IMG_SIZE = 32          # 32 or 48 or 64 (smaller = faster)
GRAYSCALE = True       # True = faster
MAX_PER_CLASS = 600    # None for no limit, or 200-1000 for speed
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15       # test will be 1 - train - val


## 3) Yardımcı fonksiyonlar

In [ ]:

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def list_class_folders(root_dir: Path):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"Not found: {root_dir}")
    class_names = []
    for p in sorted(root_dir.iterdir()):
        if p.is_dir():
            class_names.append(p.name)
    if not class_names:
        raise RuntimeError(f"No class folders found under: {root_dir}")
    return class_names

def gather_images_by_class(root_dir: Path):
    root_dir = Path(root_dir)
    class_names = list_class_folders(root_dir)
    by_class = {}
    for cname in class_names:
        cdir = root_dir / cname
        paths = []
        for p in cdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(str(p))
        by_class[cname] = paths
    return class_names, by_class

def split_list(items, train_ratio, val_ratio):
    items = list(items)
    random.shuffle(items)
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train = items[:n_train]
    val = items[n_train:n_train+n_val]
    test = items[n_train+n_val:]
    return train, val, test

def load_image_vector(path, img_size=32, grayscale=True):
    # Resize + normalize + flatten
    img = Image.open(path)
    img = img.convert("L" if grayscale else "RGB")
    img = img.resize((img_size, img_size))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if not grayscale:
        # (H,W,3) -> flatten
        arr = arr.reshape(-1)
    else:
        # (H,W) -> flatten
        arr = arr.reshape(-1)
    return arr

def build_splits_from_folder_dataset(root_dir: Path, out_npz_path: Path):
    class_names, by_class = gather_images_by_class(root_dir)

    X_train, y_train, X_val, y_val, X_test, y_test = [], [], [], [], [], []
    for idx, cname in enumerate(class_names):
        paths = by_class[cname]
        if MAX_PER_CLASS is not None and len(paths) > MAX_PER_CLASS:
            paths = random.sample(paths, MAX_PER_CLASS)

        tr, va, te = split_list(paths, TRAIN_RATIO, VAL_RATIO)

        for p in tr:
            X_train.append(load_image_vector(p, IMG_SIZE, GRAYSCALE))
            y_train.append(idx)
        for p in va:
            X_val.append(load_image_vector(p, IMG_SIZE, GRAYSCALE))
            y_val.append(idx)
        for p in te:
            X_test.append(load_image_vector(p, IMG_SIZE, GRAYSCALE))
            y_test.append(idx)

    # Convert to arrays
    X_train = np.stack(X_train).astype(np.float32)
    X_val   = np.stack(X_val).astype(np.float32)
    X_test  = np.stack(X_test).astype(np.float32)
    y_train = np.array(y_train, dtype=np.int64)
    y_val   = np.array(y_val, dtype=np.int64)
    y_test  = np.array(y_test, dtype=np.int64)

    # Standardization using train stats (helps LR)
    mean = X_train.mean(axis=0, keepdims=True)
    std  = X_train.std(axis=0, keepdims=True) + 1e-6

    X_train = (X_train - mean) / std
    X_val   = (X_val   - mean) / std
    X_test  = (X_test  - mean) / std

    np.savez_compressed(
        out_npz_path,
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test,
        class_names=np.array(class_names, dtype=object),
        img_size=IMG_SIZE,
        grayscale=int(GRAYSCALE),
        mean=mean.astype(np.float32),
        std=std.astype(np.float32),
        seed=SEED,
    )
    print(f"Saved: {out_npz_path} | train={len(X_train)} val={len(X_val)} test={len(X_test)} classes={len(class_names)}")


## 4) PlantVillage preprocess

PlantVillage klasörün içinde doğrudan sınıf klasörleri varsa çalışır.
Eğer sende farklı bir alt klasör (ör. `color/`) varsa `PLANTVILLAGE_DIR` path'ini ona göre ayarla.

In [ ]:

PV_OUT = OUT_DIR / "plantvillage_preprocessed.npz"

# Quick check: if this fails, adjust PLANTVILLAGE_DIR above
print("PlantVillage dir:", PLANTVILLAGE_DIR)
# Uncomment to preprocess:
# build_splits_from_folder_dataset(PLANTVILLAGE_DIR, PV_OUT)


## 5) PlantDoc Converted preprocess

Converted klasörün içinde doğrudan sınıf klasörleri olacak şekilde ayarlanmışsa çalışır.

In [ ]:

PD_OUT = OUT_DIR / "plantdoc_converted_preprocessed.npz"

print("PlantDoc converted dir:", PLANTDOC_CONVERTED_DIR)
# Uncomment to preprocess:
# build_splits_from_folder_dataset(PLANTDOC_CONVERTED_DIR, PD_OUT)


## 6) Plant Disease Detection preprocess

Bu dataset çoğu zaman zaten Train/Validation/Test olarak ayrılmış gelir.
Burada split işlemi yapmıyoruz; klasörlerden direkt yüklüyoruz ve yine tek `.npz` olarak kaydediyoruz.

In [ ]:

def build_from_explicit_splits(train_dir: Path, val_dir: Path, test_dir: Path, out_npz_path: Path):
    # Class names should match across splits
    class_names = list_class_folders(train_dir)

    def load_split(split_dir):
        X, y = [], []
        for idx, cname in enumerate(class_names):
            cdir = split_dir / cname
            if not cdir.exists():
                raise FileNotFoundError(f"Missing class folder: {cdir}")
            paths = [p for p in cdir.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if MAX_PER_CLASS is not None and len(paths) > MAX_PER_CLASS:
                paths = random.sample(paths, MAX_PER_CLASS)
            for p in paths:
                X.append(load_image_vector(str(p), IMG_SIZE, GRAYSCALE))
                y.append(idx)
        X = np.stack(X).astype(np.float32)
        y = np.array(y, dtype=np.int64)
        return X, y

    X_train, y_train = load_split(train_dir)
    X_val, y_val     = load_split(val_dir)
    X_test, y_test   = load_split(test_dir)

    mean = X_train.mean(axis=0, keepdims=True)
    std  = X_train.std(axis=0, keepdims=True) + 1e-6

    X_train = (X_train - mean) / std
    X_val   = (X_val   - mean) / std
    X_test  = (X_test  - mean) / std

    np.savez_compressed(
        out_npz_path,
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test,
        class_names=np.array(class_names, dtype=object),
        img_size=IMG_SIZE,
        grayscale=int(GRAYSCALE),
        mean=mean.astype(np.float32),
        std=std.astype(np.float32),
        seed=SEED,
    )
    print(f"Saved: {out_npz_path} | train={len(X_train)} val={len(X_val)} test={len(X_test)} classes={len(class_names)}")

PDD_OUT = OUT_DIR / "plant_disease_detection_preprocessed.npz"

print("PDD train dir:", PDD_TRAIN_DIR)
# Uncomment to preprocess:
# build_from_explicit_splits(PDD_TRAIN_DIR, PDD_VAL_DIR, PDD_TEST_DIR, PDD_OUT)


## 7) Notlar (yanlış sonuçları azaltmak için)

- **Label mapping**: `class_names` preprocess'te kaydediliyor; train/test her zaman bunu kullanıyor.
- **Normalization**: train'den hesaplanan `mean/std` val/test'e aynı şekilde uygulanıyor.
- **Hız**: `IMG_SIZE=32`, `GRAYSCALE=True`, `MAX_PER_CLASS` ile eğitim ciddi hızlanır.